# IMPORT LIBRARY

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

# CONNECTING TO DATABASE

In [ ]:
load_dotenv("../.env")

conn = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connection created")

# RETRIVE DATA

In [ ]:
customer_data = '''
    WITH cust_activity AS(
        SELECT 
            customer_id,
            last_interaction,
            churn_month,
            (churn_month - last_interaction) AS days_before_churn,
            churn_status,
            engagement_score
        FROM (
            SELECT
                customer_id,
                month AS last_interaction,
                ROW_NUMBER() OVER (
                    PARTITION BY customer_id 
                    ORDER BY month DESC
                    ) AS rn,
                churn_month,
                churn_status,
                AVG(engagement_score) AS engagement_score
            FROM customers_monthly_metrics
            GROUP BY
                customer_id,
                month,
                churn_month,
                churn_status) AS rn
        WHERE 
            (churn_status = 'Churned' AND rn = 2)
            OR
            (churn_status = 'Active' AND rn = 1)
        ),

    cust_activity_pra_churn AS(
        SELECT
            cm.customer_id
            ,MIN(CASE 
            WHEN i.event_ts > cm.churn_month
            THEN i.event_ts
            ELSE NULL
            END) AS return_date
            ,COUNT(i.interaction_id) AS total_interaction
        FROM customers_monthly_metrics cm
        LEFT JOIN interactions i ON cm.customer_id = i.customer_id
        GROUP BY cm.customer_id),'''